# Dimensional Data Warehouse Implementation

This notebook implements a complete star schema data warehouse pattern following Databricks best practices.

## Architecture Overview

### Dimension Tables (SCD Type 2)
* **DimCustomer**: Customer demographic information with history tracking
* **DimProduct**: Product catalog with hierarchy
* **DimDate**: Date dimension for time-based analysis
* **DimGeography**: Geographic location hierarchy

### Fact Tables
* **FactInternetSales**: Online sales transactions



## IDENTITY Column: ALWAYS vs BY DEFAULT

### GENERATED BY DEFAULT AS IDENTITY (Current Choice)
* Auto-generates values when not provided
* **Allows manual insertion** — you CAN explicitly specify values
* Useful for flexibility and data migration scenarios

**Example:**
```sql
-- Auto-generated ID
INSERT INTO DimCustomer (FirstName, LastName) VALUES ('John', 'Doe');

-- Manual ID (allowed)
INSERT INTO DimCustomer (CustomerKey, FirstName, LastName) VALUES (9999, 'Jane', 'Smith');
```

### GENERATED ALWAYS AS IDENTITY
* **Always auto-generates** values
* **Blocks manual insertion** — attempting to specify a value throws an error
* Enforces strict control over ID generation

**Example:**
```sql
-- Auto-generated ID (works)
INSERT INTO DimCustomer (FirstName, LastName) VALUES ('John', 'Doe');

-- Manual ID (ERROR!)
INSERT INTO DimCustomer (CustomerKey, FirstName, LastName) VALUES (9999, 'Jane', 'Smith');
```

### Which to Use?

**Use BY DEFAULT** when you need:
* Data migration with existing IDs
* Flexibility to manually set IDs occasionally
* Loading historical dimension data

**Use ALWAYS** when you want:
* Strict enforcement (no manual overrides)
* Guaranteed sequence-based IDs
* Maximum data integrity

## Dimension Tables

Dimension tables contain descriptive attributes for business entities. 

In [0]:
%sql

CREATE OR REPLACE TABLE DimCustomer(
    CustomerKey BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    CustomerAlternateKey STRING,
    CustomerType STRING,
    Title STRING,
    FirstName STRING,
    LastName STRING,
    MiddleName STRING,
    Suffix STRING,
    EmailRequest STRING,
    BirthDate DATE,
    MaritalStatus STRING,
    YearlyIncome STRING,
    Gender STRING,
    TotalChildren INT,
    NumberChildrenAtHome INT,
    Education STRING,
    Occupation STRING,
    IsHomeOwner INT,
    NumberCarsOwned INT,
    CommuteDistance STRING,
    StartDate TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP(),
    EndDate TIMESTAMP,
    IsLateArriving INT NOT NULL DEFAULT 0
    )
TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
%sql
-- DimProduct: Product dimension with hierarchy

CREATE OR REPLACE TABLE DimProduct(
    ProductKey BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    ProductAlternateKey STRING,
    ProductName STRING NOT NULL,
    ProductCategory STRING,
    ProductSubcategory STRING,
    ProductLine STRING,
    ProductModel STRING,
    Color STRING,
    Size STRING,
    Weight DECIMAL(10,2),
    StandardCost DECIMAL(10,2),
    ListPrice DECIMAL(10,2),
    StartDate TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP(),
    EndDate TIMESTAMP,
    IsLateArriving INT NOT NULL DEFAULT 0
)
TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
%sql
-- DimGeography: Geographic hierarchy

CREATE OR REPLACE TABLE DimGeography(
    GeographyKey BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    City STRING,
    StateProvince STRING,
    StateProvinceCode STRING,
    Country STRING,
    CountryRegion STRING,
    PostalCode STRING,
    SalesTerritoryKey INT,
    StartDate TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP(),
    EndDate TIMESTAMP,
    IsLateArriving INT NOT NULL DEFAULT 0
)
TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
%sql
-- DimDate: Date dimension for time intelligence

CREATE OR REPLACE TABLE DimDate(
    DateKey INT PRIMARY KEY,
    FullDate DATE NOT NULL,
    DayOfWeek INT,
    DayName STRING,
    DayOfMonth INT,
    DayOfYear INT,
    WeekOfYear INT,
    MonthName STRING,
    MonthOfYear INT,
    Quarter INT,
    QuarterName STRING,
    Year INT,
    IsWeekend BOOLEAN,
    IsHoliday BOOLEAN,
    FiscalYear INT,
    FiscalQuarter INT
);

## Fact Tables

Fact tables store measurable business events (sales transactions) with foreign keys to dimension tables.

In [0]:
%sql
CREATE OR REPLACE TABLE FactInternetSales (
    ProductKey BIGINT NOT NULL,
    OrderDateKey INT NOT NULL,
    DueDateKey INT NOT NULL,
    ShipDateKey INT NOT NULL,
    CustomerKey BIGINT NOT NULL,
    SalesOrderNumber BIGINT,
    SalesOrderLineNumber BIGINT,
    OrderQuantity INT,
    UnitPrice DECIMAL(12, 4),
    SalesAmount DECIMAL(12, 4),
    SalesAmountBeforeDiscount DECIMAL(12, 4),
    DiscountAmount DECIMAL(12, 4),
    LastModifiedDateTime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP(),
    CONSTRAINT factinternetsales_dimproduct_productkey_fk
      FOREIGN KEY (ProductKey) REFERENCES DimProduct (ProductKey),
    CONSTRAINT factinternetsales_dimdate_orderdatekey_fk
      FOREIGN KEY (OrderDateKey) REFERENCES DimDate (DateKey),
    CONSTRAINT factinternetsales_dimdate_duedatekey_fk
      FOREIGN KEY (DueDateKey) REFERENCES DimDate (DateKey),
    CONSTRAINT factinternetsales_dimdate_shipdatekey_fk
      FOREIGN KEY (ShipDateKey) REFERENCES DimDate (DateKey),
    CONSTRAINT factinternetsales_dimcustomer_customerkey_fk
      FOREIGN KEY (CustomerKey) REFERENCES DimCustomer (CustomerKey),
    PRIMARY KEY (SalesOrderNumber, SalesOrderLineNumber)
) CLUSTER BY (OrderDateKey)
TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported');

## Understanding Named Foreign Key Constraints

### What is `CONSTRAINT factinternetsales_dimdate_orderdatekey_fk`?

This line defines a **named foreign key constraint** that enforces referential integrity between tables.

**Syntax:**
```sql
CONSTRAINT factinternetsales_dimdate_orderdatekey_fk
  FOREIGN KEY (OrderDateKey) REFERENCES DimDate (DateKey)
```

### Purpose

1. **Enforces Referential Integrity**: Ensures every `OrderDateKey` value in `FactInternetSales` must exist as a `DateKey` in the `DimDate` table
2. **Prevents Orphaned Records**: You cannot insert a sales record with an invalid date key
3. **Provides Clear Documentation**: The name follows a convention:
   - `factinternetsales` - source table
   - `dimdate` - referenced table
   - `orderdatekey` - the column being constrained
   - `fk` - foreign key

### Benefits of Naming Constraints

**With a named constraint:**
```sql
-- Clear error messages
"Constraint factinternetsales_dimdate_orderdatekey_fk violated"

-- Easy to manage
ALTER TABLE FactInternetSales DROP CONSTRAINT factinternetsales_dimdate_orderdatekey_fk;
```

**Without a name (anonymous):**
```sql
FOREIGN KEY (OrderDateKey) REFERENCES DimDate (DateKey)
-- Gets auto-generated name like "fk_abc123xyz"
-- Harder to identify and manage
```

### Multiple Date Constraints

Notice the fact table has **three named date foreign keys**:
* `factinternetsales_dimdate_orderdatekey_fk` (OrderDateKey → DimDate)
* `factinternetsales_dimdate_duedatekey_fk` (DueDateKey → DimDate)
* `factinternetsales_dimdate_shipdatekey_fk` (ShipDateKey → DimDate)

This naming pattern makes it immediately clear which date field each constraint validates.

## Sample Data Loading

Examples of loading initial data into dimension tables.

In [0]:
%sql
-- Insert sample customers

INSERT INTO DimCustomer 
(CustomerAlternateKey, CustomerType, FirstName, LastName, EmailRequest, BirthDate, MaritalStatus, Gender, YearlyIncome, TotalChildren)
VALUES
('C001', 'Individual', 'John', 'Doe', 'john.doe@email.com', '1985-05-15', 'M', 'M', '75000', 2),
('C002', 'Individual', 'Jane', 'Smith', 'jane.smith@email.com', '1990-08-22', 'S', 'F', '85000', 0),
('C003', 'Individual', 'Robert', 'Johnson', 'robert.j@email.com', '1978-03-10', 'M', 'M', '95000', 3);

In [0]:
%sql
-- Insert sample products

INSERT INTO DimProduct 
(ProductAlternateKey, ProductName, ProductCategory, ProductSubcategory, Color, ListPrice)
VALUES
('P001', 'Mountain Bike 100', 'Bikes', 'Mountain Bikes', 'Silver', 3399.99),
('P002', 'Road Bike 200', 'Bikes', 'Road Bikes', 'Red', 2899.99),
('P003', 'Touring Bike 300', 'Bikes', 'Touring Bikes', 'Blue', 1999.99),
('P004', 'Helmet', 'Accessories', 'Helmets', 'Black', 59.99),
('P005', 'Water Bottle', 'Accessories', 'Bottles', 'Blue', 9.99);

In [0]:
%sql
-- Generate date dimension for 2024-2026

INSERT INTO DimDate
SELECT 
    CAST(date_format(date, 'yyyyMMdd') AS INT) AS DateKey,
    date AS FullDate,
    dayofweek(date) AS DayOfWeek,
    date_format(date, 'EEEE') AS DayName,
    dayofmonth(date) AS DayOfMonth,
    dayofyear(date) AS DayOfYear,
    weekofyear(date) AS WeekOfYear,
    date_format(date, 'MMMM') AS MonthName,
    month(date) AS MonthOfYear,
    quarter(date) AS Quarter,
    concat('Q', quarter(date)) AS QuarterName,
    year(date) AS Year,
    CASE WHEN dayofweek(date) IN (1, 7) THEN true ELSE false END AS IsWeekend,
    false AS IsHoliday,
    year(date) AS FiscalYear,
    quarter(date) AS FiscalQuarter
FROM (
    SELECT explode(sequence(to_date('2024-01-01'), to_date('2026-12-31'), interval 1 day)) AS date
);

In [0]:
%sql
-- Insert sample internet sales transactions

INSERT INTO FactInternetSales
(SalesOrderNumber, SalesOrderLineNumber, CustomerKey, ProductKey, OrderDateKey, DueDateKey, ShipDateKey, OrderQuantity, UnitPrice, SalesAmount, SalesAmountBeforeDiscount, DiscountAmount)
VALUES
('001', 1, 1, 1, 20260115, 20260117, 20260118, 2, 3399.99, 6799.98, 6849.98, 50.00),
('001', 2, 1, 4, 20260115, 20260117, 20260118, 1, 59.99, 59.99, 64.99, 5.00),
('002', 1, 2, 2, 20260120, 20260122, 20260123, 1, 2899.99, 2899.99, 2944.99, 45.00),
('003', 1, 3, 3, 20260125, 20260127, 20260128, 1, 1999.99, 1999.99, 2039.99, 40.00),
('003', 2, 3, 5, 20260125, 20260127, 20260128, 3, 9.99, 29.97, 32.97, 3.00);